<h1 style="font-size:30px;">Assignment 5: Exploring StyleGAN2 Latent Space</h1>

<p style="font-size:16px;"><b>Objective:</b> Study how moving through the latent space of a pretrained GAN can produce smooth changes in generated face images.</p>

<p style="font-size:15px;"><b>Model used:</b> NVIDIA's pretrained StyleGAN2-ADA model trained on the FFHQ face dataset. The model is already trained, so this notebook only performs image generation and latent-space experiments.</p>

<p style="font-size:15px;"><b>Before starting:</b> In Google Colab, select <i>Runtime → Change runtime type → GPU</i>. GPU is recommended for faster execution.</p>

<h2 style="font-size:23px;">Step 1 — Set up the StyleGAN2-ADA repository</h2>

<p style="font-size:15px;">Clone NVIDIA's official repository and install the required package.</p>

In [ ]:
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
%cd stylegan2-ada-pytorch
!pip install ninja -q


<h2 style="font-size:23px;">Step 2 — Import the required libraries</h2>

In [ ]:
import torch
import numpy as np
import dnnlib
import legacy
import matplotlib.pyplot as plt
import imageio
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Selected device:", device)


<h2 style="font-size:23px;">Step 3 — Load the pretrained FFHQ StyleGAN2 model</h2>

<p style="font-size:15px;">The pretrained model weights are loaded from NVIDIA's official source. The download may take a short while.</p>

In [ ]:
network_pkl = "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl"

with dnnlib.util.open_url(network_pkl) as f:
    G = legacy.load_network_pkl(f)['G_ema'].to(device)

print("StyleGAN2 loaded. Latent dimension (z_dim):", G.z_dim)


<h2 style="font-size:23px;">Step 4 — Create an image-generation helper</h2>

<p style="font-size:15px;">This function takes a latent vector <b>z</b> and converts it into a generated image.</p>

In [ ]:
def generate_image(z, truncation_psi=0.7):
    z = z.to(device)
    label = torch.zeros([1, G.c_dim], device=device)  # FFHQ does not require class labels
    img = G(z, label, truncation_psi=truncation_psi, noise_mode="const")
    img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    return img[0].cpu().numpy()


<h2 style="font-size:23px;">Step 5 — Create two random latent vectors</h2>

<p style="font-size:15px;">Two random latent vectors are generated and used to create two different face images.</p>

In [ ]:
torch.manual_seed(42)  # Change the seed if you want a different pair of images

z1 = torch.randn([1, G.z_dim])
z2 = torch.randn([1, G.z_dim])

img1 = generate_image(z1)
img2 = generate_image(z2)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img1)
axes[0].set_title("Generated image — z1")
axes[0].axis("off")
axes[1].imshow(img2)
axes[1].set_title("Generated image — z2")
axes[1].axis("off")
plt.show()


<h2 style="font-size:23px;">Step 6 — Interpolate between the two latent vectors</h2>

<p style="font-size:15px;">Ten intermediate latent vectors are produced between <b>z1</b> and <b>z2</b>. This helps us observe whether the generated face changes smoothly.</p>

In [ ]:
n_steps = 10
alphas = np.linspace(0, 1, n_steps)

interpolated_images = []
for alpha in alphas:
    z_interp = (1 - alpha) * z1 + alpha * z2
    img = generate_image(z_interp)
    interpolated_images.append(img)

fig, axes = plt.subplots(1, n_steps, figsize=(20, 2.5))
for i, img in enumerate(interpolated_images):
    axes[i].imshow(img)
    axes[i].axis("off")
    axes[i].set_title(f"{alphas[i]:.1f}", fontsize=8)
plt.suptitle("Latent-space interpolation: z1 → z2")
plt.tight_layout()
plt.show()


<h2 style="font-size:23px;">Step 7 — Export the interpolation results</h2>

<p style="font-size:15px;">The generated sequence is saved both as an animated GIF and as one horizontal image strip.</p>

In [ ]:
# Save as GIF
imageio.mimsave("interpolation.gif", interpolated_images, duration=0.3)
print("GIF saved as interpolation.gif")

# Save as a single horizontal image strip
strip = np.concatenate(interpolated_images, axis=1)
Image.fromarray(strip).save("interpolation_strip.png")
print("Image strip saved as interpolation_strip.png")

plt.figure(figsize=(20, 2.5))
plt.imshow(strip)
plt.axis("off")
plt.title("StyleGAN2 Interpolation Strip")
plt.show()


<h2 style="font-size:23px;">Step 8 — Test individual latent dimensions</h2>

<p style="font-size:15px;">A base latent vector is kept fixed while one dimension at a time is changed. This lets us check whether particular latent directions create visible changes such as expression, pose, lighting, or hair.</p>

In [ ]:
base_z = torch.randn([1, G.z_dim])
base_img = generate_image(base_z)

# Vary one latent dimension across several values
dim_to_change = 5  # Change this index to test another latent direction
values = np.linspace(-3, 3, 6)

modified_images = []
for v in values:
    z_mod = base_z.clone()
    z_mod[0, dim_to_change] = v
    modified_images.append(generate_image(z_mod))

fig, axes = plt.subplots(1, len(values), figsize=(15, 3))
for i, img in enumerate(modified_images):
    axes[i].imshow(img)
    axes[i].axis("off")
    axes[i].set_title(f"dim {dim_to_change} = {values[i]:.1f}", fontsize=8)
plt.suptitle(f"Latent dimension {dim_to_change}: visual effect")
plt.tight_layout()
plt.show()


In [ ]:
# Test another latent dimension for comparison
dim_to_change_2 = 20  # Choose another dimension index

modified_images_2 = []
for v in values:
    z_mod = base_z.clone()
    z_mod[0, dim_to_change_2] = v
    modified_images_2.append(generate_image(z_mod))

fig, axes = plt.subplots(1, len(values), figsize=(15, 3))
for i, img in enumerate(modified_images_2):
    axes[i].imshow(img)
    axes[i].axis("off")
    axes[i].set_title(f"dim {dim_to_change_2} = {values[i]:.1f}", fontsize=8)
plt.suptitle(f"Latent dimension {dim_to_change_2}: visual effect")
plt.tight_layout()
plt.show()


<h2 style="font-size:23px;">Step 9 — Observations</h2>

<p style="font-size:15px;"><i>Note: The exact visual result can vary depending on the random seed and the latent dimensions selected.</i></p>

<p style="font-size:16px;"><b>1. Changes during interpolation</b></p>
<p style="font-size:15px;">As the latent vector moves from z1 toward z2, the generated face changes gradually. Instead of an abrupt change, features such as pose, expression, lighting, and hair transition smoothly through the intermediate images.</p>

<p style="font-size:16px;"><b>2. Meaning of the latent space</b></p>
<p style="font-size:15px;">The latent space seems to capture higher-level visual information about a face, including attributes such as age, pose, expression, lighting, and hairstyle. These characteristics can vary continuously rather than representing individual image pixels directly.</p>

<p style="font-size:16px;"><b>3. Effect of individual dimensions</b></p>
<p style="font-size:15px;">Changing one latent dimension may cause a small or difficult-to-identify difference. In some cases, however, a dimension can produce a more noticeable change in a feature such as lighting, head direction, or hair. This suggests that some latent directions have interpretable effects, although they are not always completely separated.</p>